In [ ]:
!pip install transformers pandas openpyxl torch

In [ ]:
pip install transformers torch pandas openpyxl tqdm

In [ ]:
pip install transformers


In [ ]:
!pip install --force-reinstall transformers torch


In [ ]:

# Reinstall/upgrade transformers, torch, and torchvision to resolve potential dependency issues and CUDA version mismatch.
# This ensures a clean installation and compatible versions.
!pip install --force-reinstall transformers torch torchvision


In [ ]:
import pandas as pd
import json
import re


from transformers import pipeline

#model
print("Loading NLP Engine...")
classifier = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-3")


from google.colab import drive
drive.mount('/content/drive')




In [ ]:
import pandas as pd
df = pd.read_json('/content/drive/MyDrive/arxiv-metadata-oai-snapshot.json', lines=True, nrows=6000)
df.head()

In [ ]:

arxiv_map = {
    'cs.AI': 'Computer Science',
    'hep-ph': 'Physics',
    'physics.gen-ph': 'Physics',
    'cs.CV': 'Computer Science',
    'hep-th': 'Physics',
    'astro-ph': 'Physics',
    'quant-ph': 'Physics',
    'cs.CL': 'Computer Science',
    'cs.NE': 'Computer Science',
    'cs.LG': 'Computer Science',
    'math.CO': 'Mathematics',
    'math.CA': 'Mathematics',
    'stat.ML': 'Statistics',
    'math.NT': 'Mathematics',
    'gr-qc': 'Physics',
    'nlin.PS': 'Physics',
    'math.RA': 'Mathematics',
    'math.PR': 'Mathematics',
    'math.NA': 'Mathematics',
    'cond-mat': 'Mathematics',
    'cond-mat.mes-hall': 'Physics',
    'q-bio.PE': 'Biology',
    'hep-ex': 'Physics',
    'cond-mat.str-el': 'Mathematics',
    'q-bio.QM': 'Biology',
    'math.QA': 'Mathematics',
    'math.OA': 'Mathematics',
    'cond-mat.stat-mech': 'Mathematics',
    'cond-mat.mtrl-sci': 'Physics',
    'nucl-th': 'Physics',
    'nucl-ex': 'Physics',
    'hep-lat': 'Physics',
    'nlin.SI': 'Physics',
    'q-fin.PR': 'Economics',
    'q-fin.ST': 'Economics', # Statistical Finance
    'q-fin.RM': 'Economics', # Risk Management

}

def map_by_prefix(cat_string):
    primary = str(cat_string).split(' ')[0]
    if primary.startswith('q-bio'): return 'Biology'
    if primary.startswith('physics') or primary.startswith('astro-ph') or primary.startswith('cond-mat'): return 'Physics'
    if primary.startswith('cs'): return 'Computer Science'
    if primary.startswith('math'): return 'Mathematics'
    if primary.startswith('stat'): return 'Statistics',
    if primary.startswith('q-fin'): return 'Economics'
    return arxiv_map.get(primary, primary)



In [ ]:
df['clean_label'] = df['categories'].apply(map_by_prefix)
candidate_labels = ["Physics", "Computer Science", "Mathematics", "Biology", "Medicine", "Economics"]
print("Starting classification...")
print(f"Classifying {len(df)} articles...")


def classify_text(row):
    text = f"{row['title']}. {row['abstract']}"
    res = classifier(text, candidate_labels)
    return res['labels'][0], round(res['scores'][0] * 100, 2)

df[['predicted_topic', 'confidence']] = df.apply(
    lambda row: pd.Series(classify_text(row)), axis=1
)



In [ ]:
df.to_excel('spyder_arxiv_results.xlsx', index=False)
print("Success! File saved as 'spyder_arxiv_results.xlsx'")

def check_if_correct(row):
    predicted = str(row['predicted_topic']).lower()
    actual = str(row['clean_label']).lower()
    if predicted in actual or actual in predicted:
        return "Yes"
    else:
        return "No"
df['is_correct'] = df.apply(check_if_correct, axis=1)

accuracy = (df['is_correct'] == 'Yes').mean() * 100
print(f"Model Accuracy: {accuracy:.2f}%")

df.to_excel('spyder_arxiv_results_with_accuracy.xlsx', index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df.to_excel('/content/drive/MyDrive/results_6000_rows.xlsx', index=False)